# Flock Energy Take-Home Assignment: End-to-End Pipeline
## REFIT Smart-Meter Data Cleaning, Washing Machine EDA, Disaggregation & Practical Implications

**Author:** Data Science Candidate  
**Dataset:** REFIT Electrical Load Measurements (University of Strathclyde, Loughborough, UEA)  

---

### Table of Contents:
1. **[Part 1: Basic Inspection and Cleaning of Raw Data (House 1)](#part1)**
   - Raw telemetry merging, non-monotonic ordering, duplicate timestamps, physical limits, 1-minute resampling, and outage flagging.
2. **[Part 2: Exploratory Data Analysis of Washing-Machine Use (4 Houses)](#part2)**
   - Plain-language cycle detection rule, availability audit, cycle duration/energy cross-comparison, and diurnal/weekly patterns.
3. **[Part 3: Appliance Disaggregation Modeling & Evaluation](#part3)**
   - Chronological 80/20 train/test split, feature engineering, Always-Off and Hourly-Profile baselines vs. Gradient Boosted Decision Tree (GBDT), evaluation metrics (the NILM paradox), and failure mode analysis.
4. **[Part 4: Practical Implications, Sampling Resolution Impact & Utility Insights](#part4)**
   - Quantifying energy and bill savings from peak-shifting and eco-wash adoption, A/B testing framework, simulation of 15-min and 30-min data degradation, and domain transfer limitations from UK to India.

In [ ]:
# Environment & Library Imports
import os
import gc
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    precision_score,
    recall_score,
    f1_score
)

os.makedirs('figures', exist_ok=True)
print('Environment initialized successfully!')

<a id='part1'></a>
# Part 1: Basic Inspection and Cleaning of Raw Data (House 1)

### Objectives:
1. Load raw telemetry from `RAW_House1_Part1.csv` and `RAW_House1_Part2.csv`.
2. Transition from Part 1 to Part 2 where explicit `NaN` sensor tracking began (`2014-06-27`).
3. Identify and document data quality flaws: non-monotonic timestamps, duplicates, irregular sampling, extreme sensor spikes.
4. Apply documented cleaning rules, resample to uniform 1-minute intervals, forward-fill short gaps ($\le$ 5 min), and flag extended outages (> 5 min).
5. Plot before vs. after signals for Aggregate power, Appliance 1 (Fridge), and Appliance 5 (Washing Machine).

In [ ]:
# 1. Load Raw Part 1 and Part 2
part1_path = '../RAW_House1_Part1.csv' if os.path.exists('../RAW_House1_Part1.csv') else 'RAW_House1_Part1.csv'
part2_path = '../RAW_House1_Part2.csv' if os.path.exists('../RAW_House1_Part2.csv') else 'RAW_House1_Part2.csv'

channels = ['Aggregate'] + [f'Appliance{i}' for i in range(1, 10)]
dtype_dict = {col: 'float32' for col in channels}

df1 = pd.read_csv(part1_path, dtype=dtype_dict)
df2 = pd.read_csv(part2_path, dtype=dtype_dict)

# Switch to Part 2 where explicit NaNs replace Part 1's zero-imputation
SWITCH_TIME = '2014-06-27 21:57:50'
df1 = df1[df1['Time'] < SWITCH_TIME]
raw_combined = pd.concat([df1, df2], ignore_index=True)
del df1, df2; gc.collect()

print(f'Combined raw telemetry records: {len(raw_combined):,}')

In [ ]:
# 2. Audit Data Quality Issues
raw_combined['DateTime'] = pd.to_datetime(raw_combined['Time'])
raw_combined.drop(columns=['Time', 'Unix'], inplace=True)

is_sorted = raw_combined['DateTime'].is_monotonic_increasing
print(f'Timestamps strictly chronological initially? {is_sorted}')

dup_timestamps = raw_combined.duplicated(subset=['DateTime']).sum()
print(f'Duplicate timestamps found: {dup_timestamps:,} ({dup_timestamps / len(raw_combined):.2%})')

raw_combined.sort_values('DateTime', inplace=True)
raw_combined.reset_index(drop=True, inplace=True)

intervals = raw_combined['DateTime'].diff().dt.total_seconds().dropna()
print(f'Median sampling interval: {intervals.median():.1f} s (Max gap: {intervals.max()/3600:.1f} hrs)')
del intervals; gc.collect()

# Physical sanity checks (UK domestic 13A socket limit ~3kW)
agg_spikes = (raw_combined['Aggregate'] > 15000).sum()
app_spikes = sum((raw_combined[f'Appliance{i}'] > 4000).sum() for i in range(1, 10))
print(f'Extreme spikes to clamp: {agg_spikes:,} Aggregate (>15kW) and {app_spikes:,} Appliance (>4kW)')

# Cache raw slices for before/after visualization
START_48H, END_48H = '2013-10-22 00:00:00', '2013-10-24 00:00:00'
raw_slice = raw_combined[(raw_combined['DateTime'] >= START_48H) & (raw_combined['DateTime'] <= END_48H)].copy()
spike_raw = raw_combined[(raw_combined['DateTime'] >= '2014-01-10') & (raw_combined['DateTime'] <= '2014-01-12')].copy()

In [ ]:
# 3. Execute Cleaning Pipeline
# Rule A: Deduplicate keeping last observation (latest sensor state)
raw_combined.drop_duplicates(subset=['DateTime'], keep='last', inplace=True)

# Rule B: Clamp impossible spikes to NaN
raw_combined.loc[raw_combined['Aggregate'] > 15000, 'Aggregate'] = np.nan
raw_combined.loc[raw_combined['Aggregate'] < 0, 'Aggregate'] = np.nan
for i in range(1, 10):
    raw_combined.loc[raw_combined[f'Appliance{i}'] > 4000, f'Appliance{i}'] = np.nan
    raw_combined.loc[raw_combined[f'Appliance{i}'] < 0, f'Appliance{i}'] = np.nan

# Rule C: Resample to uniform 1-minute intervals (mean power)
raw_combined.set_index('DateTime', inplace=True)
df_1min = raw_combined[channels].resample('1min').mean()
del raw_combined; gc.collect()

# Rule D: Forward-fill short gaps (<=5 min) & flag long outages (>5 min)
df_1min_filled = df_1min.ffill(limit=5)
del df_1min; gc.collect()

df_1min_filled['outage_flag'] = df_1min_filled['Aggregate'].isna().astype(int)
outage_pct = df_1min_filled['outage_flag'].mean() * 100.0
print(f'Cleaned 1-minute records: {len(df_1min_filled):,} (Outage flagged: {outage_pct:.2f}%)')

df_1min_filled.to_parquet('house1_cleaned_1min.parquet')
print('Saved house1_cleaned_1min.parquet!')

In [ ]:
# 4. Part 1 Visualizations: Before vs. After Cleaning
clean_slice = df_1min_filled.loc[START_48H:END_48H]

fig, axes = plt.subplots(3, 2, figsize=(15, 9), sharex='col', sharey='row')
axes[0, 0].plot(raw_slice['DateTime'], raw_slice['Aggregate'], color='#1f77b4', alpha=0.7, lw=0.8)
axes[0, 0].set_title('Raw: Aggregate Power (8s interval)', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel('Power (W)')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

axes[0, 1].plot(clean_slice.index, clean_slice['Aggregate'], color='#2ca02c', lw=1.2)
axes[0, 1].set_title('Cleaned: Aggregate Power (1-Minute Resampled)', fontsize=10, fontweight='bold')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

axes[1, 0].plot(raw_slice['DateTime'], raw_slice['Appliance1'], color='#ff7f0e', alpha=0.7, lw=0.8)
axes[1, 0].set_title('Raw: Appliance 1 (Fridge - noisy)', fontsize=10, fontweight='bold')
axes[1, 0].set_ylabel('Power (W)')
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

axes[1, 1].plot(clean_slice.index, clean_slice['Appliance1'], color='#d62728', lw=1.2)
axes[1, 1].set_title('Cleaned: Appliance 1 (Fridge - 1-Minute Mean)', fontsize=10, fontweight='bold')
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

axes[2, 0].plot(raw_slice['DateTime'], raw_slice['Appliance5'], color='#9467bd', alpha=0.7, lw=0.8)
axes[2, 0].set_title('Raw: Appliance 5 (Washing Machine)', fontsize=10, fontweight='bold')
axes[2, 0].set_ylabel('Power (W)')
axes[2, 0].set_xlabel('Timestamp')
axes[2, 0].grid(True, linestyle='--', alpha=0.5)

axes[2, 1].plot(clean_slice.index, clean_slice['Appliance5'], color='#8c564b', lw=1.2)
axes[2, 1].set_title('Cleaned: Appliance 5 (Washing Machine - 1-Minute Resampled)', fontsize=10, fontweight='bold')
axes[2, 1].set_ylabel('Power (W)')
axes[2, 1].set_xlabel('Timestamp')
axes[2, 1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Problem 1: House 1 Power Comparison Before vs. After Cleaning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem1_before_after_cleaning.png', dpi=200)
plt.show()

In [ ]:
# Spike Removal & Outage Demonstration
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))
spike_clean = df_1min_filled.loc['2014-01-10':'2014-01-12']
ax1.plot(spike_raw['DateTime'], spike_raw['Aggregate'], label='Raw Aggregate (with sensor spikes)', color='orange', alpha=0.8)
ax1.plot(spike_clean.index, spike_clean['Aggregate'], label='Cleaned Aggregate (spikes removed)', color='navy', lw=1.2)
ax1.set_title('Sensor Spike Removal & Signal Smoothing', fontsize=10, fontweight='bold')
ax1.set_ylabel('Power (W)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

feb_outage = df_1min_filled.loc['2014-02-18':'2014-02-25']
ax2.plot(feb_outage.index, feb_outage['Aggregate'], label='Cleaned Aggregate Power', color='teal', lw=1.2)
ax2.fill_between(feb_outage.index, 0, 3000, where=(feb_outage['outage_flag'] == 1),
                 color='crimson', alpha=0.25, label='Flagged Outage (Gaps > 5 mins left as NaN)')
ax2.set_title('Outage Flagging vs. Blind Imputation (February 2014 REFIT Outage)', fontsize=10, fontweight='bold')
ax2.set_ylabel('Power (W)')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/problem1_outage_spike_handling.png', dpi=200)
plt.show()

<a id='part2'></a>
# Part 2: Exploratory Data Analysis of Washing-Machine Use (4 Houses)

### Objectives:
1. Define a simple, physically grounded rule in plain English to identify active washing machine cycles.
2. Audit data availability, missing intervals, and off/zero time across 4 households:
   - **House 1:** Beko WMC6140 (`Appliance 5`)
   - **House 2:** LG F1289TD (`Appliance 2`)
   - **House 3:** Washing Machine (`Appliance 6`)
   - **House 5:** AEG L99695HWD (`Appliance 3`)
3. Compare cycle characteristics (duration, peak power, energy per cycle in kWh).
4. Analyze diurnal (hour-of-day) and weekly (day-of-week) usage distributions.
5. Generate the 3 required visual plots.

In [ ]:
# 1. Plain Language Cycle Detection Rule Function
def detect_washing_cycles(power_1min, p_thresh=20.0, max_gap=5, min_dur=15, min_peak=500.0):
    """
    Detects washing machine cycles using 4 domain criteria:
    - Active Power >= 20 W (separates operational states from <5 W standby).
    - Max Gap Tolerance <= 5 mins (bridges drain/fill pauses).
    - Min Duration >= 15 mins (rejects transient door openings/clicks).
    - Min Peak >= 500 W (verifies electric heating element or spin motor activation).
    """
    times = power_1min.index
    values = power_1min.values
    cycles = []
    
    in_cycle = False
    start_idx = None
    gap_count = 0
    
    for i in range(len(values)):
        val = 0.0 if np.isnan(values[i]) else values[i]
        if val >= p_thresh:
            if not in_cycle:
                in_cycle = True
                start_idx = i
            gap_count = 0
        else:
            if in_cycle:
                gap_count += 1
                if gap_count > max_gap or i == len(values) - 1:
                    end_idx = i - gap_count
                    dur = end_idx - start_idx + 1
                    cycle_vals = values[start_idx:end_idx+1]
                    peak = np.nanmax(cycle_vals) if len(cycle_vals) > 0 else 0.0
                    if dur >= min_dur and peak >= min_peak:
                        energy_kwh = (np.nansum(cycle_vals) / 60.0) / 1000.0
                        cycles.append({
                            'start_time': times[start_idx],
                            'end_time': times[end_idx],
                            'duration_min': dur,
                            'peak_watts': peak,
                            'energy_kwh': energy_kwh,
                            'start_hour': times[start_idx].hour,
                            'day_of_week': times[start_idx].day_name()
                        })
                    in_cycle = False
                    gap_count = 0
    return pd.DataFrame(cycles)

In [ ]:
# 2. Resample and Extract Cycles across 4 Households
HOUSE_CONFIG = {
    'House 1': {'file': '../CLEAN_House1.csv', 'wm_col': 'Appliance5', 'model': 'Beko WMC6140'},
    'House 2': {'file': '../CLEAN_House2.csv', 'wm_col': 'Appliance2', 'model': 'LG F1289TD'},
    'House 3': {'file': '../CLEAN_House3.csv', 'wm_col': 'Appliance6', 'model': 'Washing Machine'},
    'House 5': {'file': '../CLEAN_House5.csv', 'wm_col': 'Appliance3', 'model': 'AEG L99695HWD'}
}

house_summaries = []
all_detected_cycles = []
house_resampled_dict = {}

for h_name, cfg in HOUSE_CONFIG.items():
    f_path = cfg['file'] if os.path.exists(cfg['file']) else cfg['file'].replace('../', '')
    df = pd.read_csv(f_path, usecols=['Time', 'Aggregate', cfg['wm_col']],
                     dtype={'Aggregate': 'float32', cfg['wm_col']: 'float32'})
    df['DateTime'] = pd.to_datetime(df['Time'])
    df.set_index('DateTime', inplace=True)
    df.drop(columns=['Time'], inplace=True)
    
    s_1min = df.resample('1min').mean()
    s_1min['agg_clean'] = s_1min['Aggregate'].ffill(limit=5)
    s_1min['wm_clean'] = s_1min[cfg['wm_col']].ffill(limit=5)
    house_resampled_dict[h_name] = s_1min[['agg_clean', 'wm_clean']].copy()
    
    total_mins = len(s_1min)
    missing_pct = (s_1min['wm_clean'].isna().sum() / total_mins) * 100.0
    valid_wm = s_1min['wm_clean'].dropna()
    off_pct = ((valid_wm < 20.0).sum() / len(valid_wm)) * 100.0
    
    cycles_df = detect_washing_cycles(s_1min['wm_clean'])
    cycles_df['house'] = h_name
    all_detected_cycles.append(cycles_df)
    
    duration_days = (s_1min.index.max() - s_1min.index.min()).total_seconds() / 86400.0
    cycles_per_week = (len(cycles_df) / duration_days) * 7.0
    
    house_summaries.append({
        'House': h_name,
        'Appliance Model': cfg['model'],
        'Days Monitored': round(duration_days, 1),
        'Missing Data (%)': round(missing_pct, 2),
        'Off/Zero Time (%)': round(off_pct, 2),
        'Total Cycles': len(cycles_df),
        'Cycles / Week': round(cycles_per_week, 2),
        'Median Duration (min)': round(cycles_df['duration_min'].median(), 1),
        'Median Peak Power (W)': round(cycles_df['peak_watts'].median(), 1),
        'Median Energy (kWh)': round(cycles_df['energy_kwh'].median(), 3)
    })
    del df, s_1min; gc.collect()

summary_table = pd.DataFrame(house_summaries)
all_cycles_df = pd.concat(all_detected_cycles, ignore_index=True)
display(summary_table)

In [ ]:
# 3. Part 2 Visualizations: 24-Hour Trace, Zoomed Cycles, and Cross-Household Boxplots
trace_data = house_resampled_dict['House 1'].loc['2013-10-23 00:00:00':'2013-10-23 23:59:00']

# Plot (i): Representative 24-hour trace
plt.figure(figsize=(14, 4.5))
plt.plot(trace_data.index, trace_data['agg_clean'], label='Total Home Aggregate Power', color='#1f77b4', lw=1.2, alpha=0.8)
plt.plot(trace_data.index, trace_data['wm_clean'], label='Washing Machine Power (Beko WMC6140)', color='#d62728', lw=1.8)
plt.fill_between(trace_data.index, 0, trace_data['wm_clean'], color='#d62728', alpha=0.2)
plt.title('Plot (i): Representative 24-Hour Power Trace (House 1 - October 23, 2013)', fontsize=11, fontweight='bold')
plt.ylabel('Power (W)')
plt.xlabel('Time of Day')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figures/problem2_24h_trace.png', dpi=200)
plt.show()

# Plot (ii): Zoomed-in cycle trace
cycle_zoom = trace_data.loc['2013-10-23 05:45:00':'2013-10-23 07:30:00']
plt.figure(figsize=(12, 4.5))
plt.plot(cycle_zoom.index, cycle_zoom['wm_clean'], color='#d62728', lw=2, marker='o', markersize=3, label='Washing Machine Load')
plt.axhline(20, color='gray', linestyle=':', label='Active Threshold (20 W)')
plt.axhline(500, color='purple', linestyle='--', alpha=0.7, label='Peak Qualification (500 W)')
h1_sample_cycles = all_cycles_df[(all_cycles_df['house'] == 'House 1') & 
                                  (all_cycles_df['start_time'] >= '2013-10-23 05:45:00') & 
                                  (all_cycles_df['start_time'] <= '2013-10-23 07:30:00')]
for idx, c in h1_sample_cycles.iterrows():
    plt.axvspan(c['start_time'], c['end_time'], color='green', alpha=0.15)
    plt.text(c['start_time'], c['peak_watts'] + 50, f"{c['duration_min']}m | {c['peak_watts']:.0f}W", 
             fontsize=9, fontweight='bold', color='darkgreen')
plt.title('Plot (ii): Detected Washing Machine Operating Cycles (House 1 Zoom)', fontsize=11, fontweight='bold')
plt.ylabel('Power (W)')
plt.xlabel('Timestamp')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figures/problem2_detected_cycles.png', dpi=200)
plt.show()

In [ ]:
# Plot (iii): Cross-House Comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
h_list = ['House 1', 'House 2', 'House 3', 'House 5']
dur_data = [all_cycles_df[all_cycles_df['house'] == h]['duration_min'] for h in h_list]
axes[0, 0].boxplot(dur_data, tick_labels=h_list, showmeans=True, patch_artist=True,
                   boxprops=dict(facecolor='#c6dbef', color='#08519c'), medianprops=dict(color='red', lw=2))
axes[0, 0].set_title('A. Cycle Duration Across Households', fontsize=10, fontweight='bold')
axes[0, 0].set_ylabel('Duration (Minutes)')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

en_data = [all_cycles_df[all_cycles_df['house'] == h]['energy_kwh'] for h in h_list]
axes[0, 1].boxplot(en_data, tick_labels=h_list, showmeans=True, patch_artist=True,
                   boxprops=dict(facecolor='#c7e9c0', color='#006d2c'), medianprops=dict(color='red', lw=2))
axes[0, 1].set_title('B. Energy per Cycle Across Households', fontsize=10, fontweight='bold')
axes[0, 1].set_ylabel('Energy (kWh)')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

for h in h_list:
    h_c = all_cycles_df[all_cycles_df['house'] == h]
    hour_pct = (h_c['start_hour'].value_counts().reindex(range(24), fill_value=0) / len(h_c)) * 100.0
    axes[1, 0].plot(range(24), hour_pct, marker='o', label=h, lw=1.5)
axes[1, 0].set_title('C. Usage by Hour of Day (% of Total)', fontsize=10, fontweight='bold')
axes[1, 0].set_xlabel('Hour of Day (0-23)')
axes[1, 0].set_ylabel('Frequency (%)')
axes[1, 0].set_xticks(range(0, 24, 2))
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend()

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for h in h_list:
    h_c = all_cycles_df[all_cycles_df['house'] == h]
    day_pct = (h_c['day_of_week'].value_counts().reindex(days, fill_value=0) / len(h_c)) * 100.0
    axes[1, 1].plot(range(7), day_pct, marker='s', label=h, lw=1.5)
axes[1, 1].set_title('D. Usage by Day of Week (% of Total)', fontsize=10, fontweight='bold')
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend()

plt.suptitle('Plot (iii): Cross-House Usage & Characteristic Signatures', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem2_cross_house_comparison.png', dpi=200)
plt.show()

<a id='part3'></a>
# Part 3: Appliance Disaggregation Modeling & Evaluation

### Objectives:
1. Estimate washing machine load directly from household aggregate power at 1-minute resolution.
2. Apply a strict **chronological 80/20 train/test split** to avoid time-series data leakage.
3. Train two simple benchmarks: *Always-Off Baseline* and *Hourly Usage-Profile Baseline*.
4. Train a Gradient Boosted Decision Tree (GBDT) model using contextual rolling features.
5. Analyze the **NILM Evaluation Paradox** (why MAE alone is misleading when appliances are idle >95% of the time, and why RMSE, F1, and Energy Accuracy are necessary).
6. Plot actual vs. predicted evaluation traces and analyze typical failure cases.

In [ ]:
# 1. Feature Engineering & Chronological Split
h1_df = pd.read_parquet('house1_cleaned_1min.parquet')
h1_valid = h1_df[h1_df['outage_flag'] == 0].copy()

feat_df = pd.DataFrame(index=h1_valid.index)
feat_df['agg'] = h1_valid['Aggregate']
feat_df['agg_diff'] = h1_valid['Aggregate'].diff().fillna(0.0)
feat_df['agg_roll_mean_5'] = h1_valid['Aggregate'].rolling(5, min_periods=1).mean()
feat_df['agg_roll_mean_15'] = h1_valid['Aggregate'].rolling(15, min_periods=1).mean()
feat_df['agg_roll_max_15'] = h1_valid['Aggregate'].rolling(15, min_periods=1).max()
feat_df['hour'] = h1_valid.index.hour
feat_df['dayofweek'] = h1_valid.index.dayofweek
feat_df['target'] = h1_valid['Appliance5']
feat_df.dropna(inplace=True)

feature_cols = ['agg', 'agg_diff', 'agg_roll_mean_5', 'agg_roll_mean_15', 'agg_roll_max_15', 'hour', 'dayofweek']

# Strict 80/20 chronological split
split_idx = int(len(feat_df) * 0.80)
train_df = feat_df.iloc[:split_idx]
test_df = feat_df.iloc[split_idx:]

X_train, y_train = train_df[feature_cols], train_df['target'].values
X_test, y_test = test_df[feature_cols], test_df['target'].values

print(f'Train: {len(train_df):,} mins ({train_df.index.min().strftime("%Y-%m-%d")} to {train_df.index.max().strftime("%Y-%m-%d")})')
print(f'Test:  {len(test_df):,} mins ({test_df.index.min().strftime("%Y-%m-%d")} to {test_df.index.max().strftime("%Y-%m-%d")})')

In [ ]:
# 2. Model Training & Evaluation Benchmark
# Baseline 1: Always Off
pred_zero = np.zeros(len(y_test))

# Baseline 2: Hour of Day Mean Profile
hourly_means = train_df.groupby('hour')['target'].mean().to_dict()
pred_profile = test_df['hour'].map(hourly_means).values

# Disaggregation Model (GBDT)
disagg_model = HistGradientBoostingRegressor(max_iter=100, max_depth=8, learning_rate=0.1, random_state=42)
disagg_model.fit(X_train, y_train)
pred_model = np.clip(disagg_model.predict(X_test), 0.0, None)

def evaluate_preds(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    t_act = (y_true >= 20.0).astype(int)
    p_act = (y_pred >= 20.0).astype(int)
    prec = precision_score(t_act, p_act, zero_division=0)
    rec = recall_score(t_act, p_act, zero_division=0)
    f1 = f1_score(t_act, p_act, zero_division=0)
    act_kwh = (np.sum(y_true) / 60.0) / 1000.0
    est_kwh = (np.sum(y_pred) / 60.0) / 1000.0
    acc = max(0.0, 1.0 - abs(est_kwh - act_kwh) / (act_kwh + 1e-6)) * 100.0
    return {
        'Model': name,
        'MAE (W)': round(mae, 2),
        'RMSE (W)': round(rmse, 2),
        'Precision': round(prec, 3),
        'Recall': round(rec, 3),
        'F1-Score': round(f1, 3),
        'Actual Energy (kWh)': round(act_kwh, 2),
        'Est. Energy (kWh)': round(est_kwh, 2),
        'Energy Acc (%)': round(acc, 1)
    }

eval_table = pd.DataFrame([
    evaluate_preds(y_test, pred_zero, '1. Always-Off Baseline'),
    evaluate_preds(y_test, pred_profile, '2. Usage-Profile Baseline'),
    evaluate_preds(y_test, pred_model, '3. Disaggregation Model (GBDT)')
])
display(eval_table)

In [ ]:
# 3. Part 3 Visualizations: Actual vs. Predicted Trace and Failure Analysis
test_results = pd.DataFrame(index=test_df.index)
test_results['Aggregate'] = test_df['agg']
test_results['Actual_WM'] = y_test
test_results['Predicted_WM'] = pred_model

active_idx = test_results[test_results['Actual_WM'] > 500].index[20].strftime('%Y-%m-%d')
eval_slice = test_results.loc[f'{active_idx} 00:00:00':pd.to_datetime(active_idx) + pd.Timedelta(days=2)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7.5), sharex=True)
ax1.plot(eval_slice.index, eval_slice['Aggregate'], label='Household Aggregate Power', color='gray', alpha=0.5, lw=1)
ax1.plot(eval_slice.index, eval_slice['Actual_WM'], label='Actual Washing Machine (Ground Truth)', color='#1f77b4', lw=2)
ax1.plot(eval_slice.index, eval_slice['Predicted_WM'], label='Predicted Washing Machine (Model)', color='#d62728', lw=1.6, linestyle='--')
ax1.set_title(f'Actual vs. Predicted Washing Machine Load ({active_idx} Evaluation Window)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Power (W)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.fill_between(eval_slice.index, 0, eval_slice['Actual_WM'], color='#1f77b4', alpha=0.25, label='Actual Energy (Ground Truth)')
ax2.plot(eval_slice.index, eval_slice['Actual_WM'], color='#1f77b4', lw=1.5)
ax2.plot(eval_slice.index, eval_slice['Predicted_WM'], color='#d62728', lw=1.5, label='Model Prediction')
ax2.set_title('Direct Disaggregation Overlay & Cycle Capture', fontsize=10, fontweight='bold')
ax2.set_ylabel('Power (W)')
ax2.set_xlabel('Timestamp')
ax2.legend(loc='upper right')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/problem3_actual_vs_predicted.png', dpi=200)
plt.show()

In [ ]:
# Failure Case Zoom: Success vs. Overlapping Signature False Positive
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4.5))
peak_time = eval_slice[eval_slice['Actual_WM'] > 1000].index[0]
zoom_success = eval_slice.loc[peak_time - pd.Timedelta(minutes=30):peak_time + pd.Timedelta(minutes=60)]
ax1.plot(zoom_success.index, zoom_success['Actual_WM'], label='Actual WM', color='#1f77b4', lw=2)
ax1.plot(zoom_success.index, zoom_success['Predicted_WM'], label='Predicted WM', color='#d62728', lw=2, linestyle='--')
ax1.plot(zoom_success.index, zoom_success['Aggregate'], label='Aggregate Power', color='gray', alpha=0.4, lw=1)
ax1.set_title('Success Case: High Peak Capture During Water Heating', fontsize=10, fontweight='bold')
ax1.set_ylabel('Power (W)')
ax1.legend(loc='upper right')
ax1.grid(True, linestyle='--', alpha=0.5)

idle_spikes = test_results[(test_results['Aggregate'] > 1800) & (test_results['Actual_WM'] == 0)]
if len(idle_spikes) > 0:
    sp_time = idle_spikes.index[5]
    zoom_fail = test_results.loc[sp_time - pd.Timedelta(minutes=30):sp_time + pd.Timedelta(minutes=45)]
    ax2.plot(zoom_fail.index, zoom_fail['Aggregate'], label='Aggregate (Kettle/Heater)', color='gray', lw=1.2)
    ax2.plot(zoom_fail.index, zoom_fail['Actual_WM'], label='Actual WM (0 W)', color='#1f77b4', lw=2)
    ax2.plot(zoom_fail.index, zoom_fail['Predicted_WM'], label='Predicted WM (Ghost Surge)', color='#d62728', lw=2, linestyle='--')
    ax2.set_title('Failure Case: Overlapping Resistive Load False Positive', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Power (W)')
    ax2.legend(loc='upper right')
    ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('figures/problem3_failure_analysis.png', dpi=200)
plt.show()

<a id='part4'></a>
# Part 4: Practical Implications, Sampling Resolution Impact & Utility Insights

### Objectives:
1. **Energy-Saving Program Simulation:** Quantify financial and carbon reductions from peak-shifting (18:00–20:00 to morning off-peak) and 30°C eco-wash adoption.
2. **A/B Testing Measurement Framework:** Formulate a rigorous randomized control trial with Difference-in-Differences (DiD) evaluation.
3. **Sampling Resolution Degradation (1-min vs. 15-min vs. 30-min):** Numerically simulate and plot how coarser smart-meter intervals dilute heating spikes and compromise disaggregation.
4. **Domain Transfer Limitations (UK to India):** Analyze the fundamental electrical differences (heated front-load vs. unheated cold-fill top-load machines).

In [ ]:
# 1. Energy-Saving Program Simulation
annual_hot_kwh = (4 * 52) * 0.613  # House 2 hot wash
annual_eco_kwh = (4 * 52) * 0.309  # House 1 eco wash
kwh_saved_home = annual_hot_kwh - annual_eco_kwh

cost_hot_peak = annual_hot_kwh * 0.35     # 35p/kWh peak
cost_eco_offpeak = annual_eco_kwh * 0.12  # 12p/kWh off-peak
savings_home = cost_hot_peak - cost_eco_offpeak

cohort_50k = 50000
total_mwh_saved = (kwh_saved_home * cohort_50k) / 1000.0
total_gbp_saved = savings_home * cohort_50k

print(f'Household Energy Saved:   {kwh_saved_home:.1f} kWh/year (-49.6%)')
print(f'Annual Bill Savings:      £{savings_home:.2f}/year per home')
print(f'Pilot (50,000 Homes):     {total_mwh_saved:,.1f} MWh/year saved (£{total_gbp_saved:,.2f})')

In [ ]:
# 2. Sampling Resolution Simulation (1-min vs. 15-min vs. 30-min)
h1_sample = h1_df.loc['2013-10-23 05:00:00':'2013-10-23 09:00:00'].copy()
s_1m = h1_sample['Appliance5']
s_15m = s_1m.resample('15min').mean()
s_30m = s_1m.resample('30min').mean()

a_1m = h1_sample['Aggregate']
a_15m = a_1m.resample('15min').mean()
a_30m = a_1m.resample('30min').mean()

print(f'1-Min Peak:  {s_1m.max():.1f} W')
print(f'15-Min Peak: {s_15m.max():.1f} W (-{1 - s_15m.max()/s_1m.max():.1%} dilution)')
print(f'30-Min Peak: {s_30m.max():.1f} W (-{1 - s_30m.max()/s_1m.max():.1%} dilution)')

fig, axes = plt.subplots(3, 1, figsize=(14, 8.5), sharex=True)
axes[0].plot(h1_sample.index, a_1m, label='Aggregate Mains Power', color='gray', alpha=0.5, lw=1)
axes[0].plot(h1_sample.index, s_1m, label='Washing Machine (1-Min)', color='#d62728', lw=1.8)
axes[0].set_title('A. 1-Minute Resolution: Sharp 2,368 W Heating Spikes & Precise Boundaries', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Power (W)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend(loc='upper right')

axes[1].step(s_15m.index, a_15m, where='post', label='Aggregate Mains (15-Min)', color='gray', alpha=0.5, lw=1)
axes[1].step(s_15m.index, s_15m, where='post', label='Washing Machine (15-Min)', color='#ff7f0e', lw=2)
axes[1].set_title(f'B. 15-Minute Resolution: Peak Flattened to {s_15m.max():.0f} W (-63% Dilution)', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Power (W)')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend(loc='upper right')

axes[2].step(s_30m.index, a_30m, where='post', label='Aggregate Mains (30-Min)', color='gray', alpha=0.5, lw=1)
axes[2].step(s_30m.index, s_30m, where='post', label='Washing Machine (30-Min)', color='#2ca02c', lw=2)
axes[2].set_title(f'C. 30-Minute Resolution: Peak Flattened to {s_30m.max():.0f} W (-79% Dilution), Blends into Baseload', fontsize=10, fontweight='bold')
axes[2].set_ylabel('Power (W)')
axes[2].set_xlabel('Time of Day (October 23, 2013)')
axes[2].grid(True, linestyle='--', alpha=0.5)
axes[2].legend(loc='upper right')

plt.suptitle('Impact of Smart-Meter Data Resolution on Disaggregation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/problem4_sampling_resolution_impact.png', dpi=200)
plt.show()